# Episodic Strategy로 AgentCore Memory를 사용하는 LangGraph

## 소개

이 Notebook에서는 LangGraph framework를 사용하는 대화형 AI agent에 **episodic memory strategy**로 Amazon Bedrock AgentCore Memory를 통합하는 방법을 살펴봅니다. 전체 대화 session을 포착하는 episodic strategy를 중심으로, agent가 특정 식단 계획 episode를 기억하고 시간에 따른 식습관 변화를 추적하도록 구성합니다.

## 튜토리얼 세부 정보

| 항목                | 세부 정보                                                                        |
|:--------------------|:---------------------------------------------------------------------------------|
| 튜토리얼 유형       | 장기 대화형                                                                       |
| Agent 사용 사례     | Episodic Memory Strategy를 사용하는 영양 도우미                                   |
| Agentic Framework   | LangGraph                                                                        |
| LLM model           | Anthropic Claude Sonnet 3.7                                                     |
| 튜토리얼 구성 요소  | AgentCore Memory, Episodic Strategy, LangGraph Hooks, session 기반 episode       |
| 예제 난이도         | 중급                                                                              |

다음 내용을 학습합니다.
- Episodic memory strategy로 AgentCore Memory 생성
- 자동 메모리 저장을 위한 pre/post model hook 구현
- 식단 계획 session을 기억하는 영양 도우미 구축
- 이전 대화를 검색하고 성찰
- 시간에 따른 식습관 추적

### 시나리오 배경

이 예제에서는 episodic memory strategy를 사용하여 전체 식단 계획 session을 기억하는 **영양 도우미**를 만듭니다. Agent는 recipe 논의, 재료 대체, 식사 feedback을 포함한 전체 대화 episode를 포착합니다. 이를 통해 "What did I plan last week?" 같은 시간 기반 질의와 식습관 pattern 분석이 가능합니다.

## 아키텍처

<div style="text-align:left">
    <img src="architecture_episodic.png" width="65%" />
</div>

### 영양 도우미에 Episodic Memory Strategy를 사용하는 이유

- **Session 기반**: 각 식단 계획 대화가 하나의 episode가 됨
- **시간적 맥락**: 식사를 특정 시간이나 상황과 연결
- **Pattern 학습**: 선호도 변화를 추적
- **풍부한 회상**: 이전 추천의 전체 맥락을 기억

### Episodic Memory Strategy의 작동 방식

Episodic strategy는 상호 작용을 구조화된 episode로 포착하고 여러 episode를 성찰하여 유용한 insight를 생성하도록 설계되었습니다. 이 strategy는 발생한 일뿐만 아니라 각 episode의 의도, 생각, 결과도 기록합니다.

#### Episodic Strategy의 세 단계

1. **Extraction**: 단기 메모리에서 유용한 insight를 식별하여 장기 메모리에 memory record로 저장
2. **Consolidation**: 유용한 정보를 새 record에 기록할지 기존 record에 기록할지 결정
3. **Reflection**: Agent 상호 작용의 여러 episode에서 insight 생성

#### Strategy 출력

**Episodes** (XML 형식):
- 상황, 의도, 평가, 근거, episode 수준의 성찰로 구분
- 상호 작용이 진행되는 동안 turn 단위로 분석
- Operation 순서와 tool 사용 방식 파악에 도움

**Reflections** (background에서 생성):
- 여러 episode의 정보를 통합
- 다음 항목을 식별하는 폭넓은 insight 추출
  - 성공적인 strategy와 pattern
  - 잠재적인 개선 사항
  - 일반적인 실패 양상
  - 여러 상호 작용에서 얻은 교훈

#### 영양 도우미에서의 활용

- **Episodes**: 각 식단 계획 session(논의한 recipe, 재료, 결정 사항)
- **Reflections**: 식습관, 좋아하는 요리, 요리 실력의 향상 과정
- **Turn 단위**: Recipe 탐색 → 재료 질문 → 대체 재료 → 최종 선택

## 사전 요구 사항

- Python 3.10+
- 적절한 권한이 있는 AWS account
- AgentCore Memory에 필요한 권한이 있는 AWS IAM role
- Amazon Bedrock model에 대한 액세스

환경을 설정하며 시작해 보겠습니다!


In [ ]:
# https://github.com/langchain-ai/langchain-aws에서 필요한 library 설치
%pip install -qr requirements.txt

In [ ]:
import os
import logging

# LangGraph 및 LangChain component import
from langchain.chat_models import init_chat_model
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.runnables import RunnableConfig
from langgraph.store.base import BaseStore
import uuid


region = os.getenv("AWS_REGION", "us-east-1")
logging.getLogger("nutrition-agent").setLevel(logging.DEBUG)

In [ ]:
# Store로 사용할 AgentCoreMemoryStore import
from langgraph_checkpoint_aws import AgentCoreMemoryStore

# 이 예제에서는 맥락을 저장하는 데 InMemorySaver를 사용합니다.
# Production 환경에서는 memory store와 원활하게 연동되는 AgentCoreMemorySaver를 checkpointer로 사용하는 것을 적극 권장합니다.
# from langgraph_checkpoint_aws import AgentCoreMemorySaver
from langgraph.checkpoint.memory import InMemorySaver
from bedrock_agentcore.memory import MemoryClient

In [ ]:
import boto3
import json

# Memory 실행용 IAM role 생성
iam_client = boto3.client("iam")
sts_client = boto3.client("sts")
account_id = sts_client.get_caller_identity()["Account"]

ROLE_NAME = "AgentCoreMemoryExecutionRole"

# AgentCore Memory용 trust policy (gamma endpoint)
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {
                "Service": [
                    "preprod.genesis-service.aws.internal",
                    "bedrock-agentcore.amazonaws.com",
                    "developer.genesis-service.aws.internal",
                ]
            },
            "Action": "sts:AssumeRole",
        }
    ],
}

# Bedrock model 호출 권한
permissions_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": ["bedrock:InvokeModel", "bedrock:InvokeModelWithResponseStream"],
            "Resource": [
                "arn:aws:bedrock:*::foundation-model/*",
                "arn:aws:bedrock:*:*:inference-profile/*",
            ],
        }
    ],
}

try:
    # 기존 role 조회 시도
    role = iam_client.get_role(RoleName=ROLE_NAME)
    MEMORY_EXECUTION_ROLE_ARN = role["Role"]["Arn"]
    print(f"✅ Using existing role: {MEMORY_EXECUTION_ROLE_ARN}")
except iam_client.exceptions.NoSuchEntityException:
    # Role 생성
    print(f"Creating IAM role: {ROLE_NAME}")
    role = iam_client.create_role(
        RoleName=ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description="Execution role for AgentCore Memory with custom strategies",
    )
    MEMORY_EXECUTION_ROLE_ARN = role["Role"]["Arn"]

    # Inline policy 연결
    iam_client.put_role_policy(
        RoleName=ROLE_NAME,
        PolicyName="BedrockModelAccess",
        PolicyDocument=json.dumps(permissions_policy),
    )
    print(f"✅ Created role: {MEMORY_EXECUTION_ROLE_ARN}")
    print("⏳ Waiting 10 seconds for IAM propagation...")
    import time

    time.sleep(10)

print(f"\nRole ARN: {MEMORY_EXECUTION_ROLE_ARN}")

In [ ]:
memory_name = "NutritionAssistantEpisodic"
client = MemoryClient(region_name=region)
MODEL_ID = "us.anthropic.claude-3-7-sonnet-20250219-v1:0"

override_strategy = {
    "customMemoryStrategy": {
        "name": "NutritionEpisodicExtractor",
        "description": "Nutrition assistant with episodic memory for meal planning insights",
        "namespaceTemplates": ["/nutrition/{actorId}/{sessionId}/"],
        "configuration": {
            "episodicOverride": {
                "extraction": {
                    "modelId": MODEL_ID,
                    "appendToPrompt": "Extract meal planning conversations including recipes discussed, ingredients mentioned, dietary considerations, and user feedback.",
                },
                "consolidation": {
                    "modelId": MODEL_ID,
                    "appendToPrompt": "Consolidate meal planning sessions into episodes, capturing the flow of recipe exploration and decision-making.",
                },
                "reflection": {
                    "modelId": MODEL_ID,
                    "appendToPrompt": "Generate insights about dietary patterns, favorite recipes, and how meal preferences evolve over time.",
                    "namespaceTemplates": ["/nutrition/{actorId}/"],
                },
            }
        },
    }
}

memory = client.create_or_get_memory(
    name=memory_name,
    description="Nutrition assistant with episodic memory for meal planning sessions",
    memory_execution_role_arn=MEMORY_EXECUTION_ROLE_ARN,
    strategies=[override_strategy],
)
memory_id = memory["id"]

print(f"✅ Created episodic memory: {memory_id}")

### Memory 구성 개요

AgentCore Episodic Memory 설정은 다음으로 구성됩니다.

- **Extraction**: Recipe, 재료, feedback을 포함한 식단 계획 대화 포착
- **Consolidation**: 대화를 식단 계획 episode로 그룹화
- **Reflection**: 시간에 따른 식습관과 선호도에 관한 insight 생성
- **Namespaces**: 사용자별로 episode 구성 (`/nutrition/{actorId}/`)

각 대화 session은 검색하고 분석할 수 있는 episode가 됩니다.

## 3단계: Memory Store 및 LLM 초기화

이제 AgentCore Memory Store와 language model을 초기화합니다.

In [ ]:
# 장기 메모리 저장 및 검색을 활성화하도록 store 초기화
store = AgentCoreMemoryStore(memory_id=memory_id, region_name=region)

# Bedrock LLM 초기화
llm = init_chat_model(MODEL_ID, model_provider="bedrock_converse", region_name=region)

## 4단계: Memory Hook 구현

메모리 저장을 자동으로 처리하는 pre/post model hook을 만듭니다.

- **Pre-model hook**: LLM 호출 전에 user message 저장
- **Post-model hook**: LLM 호출 후 assistant 응답 저장

### Memory 처리 방식

1. Message가 actor_id 및 session_id와 함께 AgentCore Memory에 저장됩니다.
2. Episodic strategy가 대화를 처리하여 구조화된 episode를 생성합니다.
3. Episode가 turn 단위 분석 결과와 함께 `/nutrition/{actorId}/{sessionId}/` namespace에 저장됩니다.
4. 여러 episode에 걸쳐 reflection이 생성되고 `/nutrition/{actorId}/` namespace에 저장됩니다.
5. 각 episode는 상황, 의도, 평가, 대화 흐름을 포착합니다.

**참고**: Episode와 reflection으로 올바르게 처리될 수 있도록 store 내부에서 LangChain message type을 AgentCore Memory message type으로 변환합니다.


In [ ]:
def pre_model_hook(state, config: RunnableConfig, *, store: BaseStore):
    """최신 사용자 메시지를 저장하기 위해 LLM 호출 전에 실행되는 훅입니다."""
    actor_id = config["configurable"]["actor_id"]
    thread_id = config["configurable"]["thread_id"]
    # Runtime에 전달된 actor와 session 조합에 message 저장
    namespace = (actor_id, thread_id)

    messages = state.get("messages", [])
    # LLM 호출 전에 확인한 마지막 human message 저장
    for msg in reversed(messages):
        if isinstance(msg, HumanMessage):
            store.put(namespace, str(uuid.uuid4()), {"message": msg})
            break

    # Episodic strategy에서는 message만 저장하며 검색은 필요하지 않음
    # Episode와 reflection은 background에서 자동으로 생성됨
    return {"messages": messages}


def post_model_hook(state, config: RunnableConfig, *, store: BaseStore):
    """어시스턴트 응답을 저장하기 위해 LLM 호출 후에 실행되는 훅입니다."""
    actor_id = config["configurable"]["actor_id"]
    thread_id = config["configurable"]["thread_id"]

    # Runtime에 전달된 actor와 session 조합에 message 저장
    namespace = (actor_id, thread_id)

    messages = state.get("messages", [])
    # LLM 응답을 AgentCore Memory에 저장
    for msg in reversed(messages):
        if isinstance(msg, AIMessage):
            store.put(namespace, str(uuid.uuid4()), {"message": msg})
            break

    return {"messages": messages}

## 5단계: LangGraph Agent 생성

이제 memory hook을 통합한 LangGraph의 `create_react_agent`를 사용하여 영양 도우미 agent를 만듭니다. Tool node에는 장기 메모리 검색 tool만 포함하며, pre/post model hook은 인수로 지정합니다.

**참고**: 사용자 지정 agent 구현에서는 이 패턴을 따르는 모든 workflow에서 필요에 따라 Store와 tool이 실행되도록 구성할 수 있습니다. Pre/post model hook을 사용하거나 마지막에 전체 대화를 저장하는 등의 방식이 가능합니다.

In [ ]:
graph = create_react_agent(
    llm,
    store=store,
    tools=[],  # 이 예제에는 추가 tool이 필요하지 않음
    checkpointer=InMemorySaver(),  # 대화 state 관리용
    pre_model_hook=pre_model_hook,  # LLM 호출 전에 user message 저장
    post_model_hook=post_model_hook,  # LLM 호출 후 episodic 처리를 위해 assistant 응답 저장
)

## 6단계: Agent Runtime 구성

사용자와 session을 구분하는 고유 식별자로 agent를 구성해야 합니다. 이 ID는 메모리 구성과 검색에 매우 중요합니다.

### Graph 호출 입력
가장 최근의 user message만 `inputs` 인수로 전달하면 됩니다. 다른 state 변수도 포함할 수 있지만, 간단한 `create_react_agent`에서는 message만 필요합니다.

### LangGraph RuntimeConfig
LangGraph에서 config는 user ID나 session ID처럼 호출 시 필요한 속성을 포함하는 `RuntimeConfig`입니다. `AgentCoreMemorySaver`를 사용하려면 config에 `thread_id`와 `actor_id`를 설정해야 합니다. 예를 들어 AgentCore 호출 endpoint에서 호출자의 identity 또는 user ID를 기반으로 이 값을 할당할 수 있습니다. 자세한 내용은 [여기에서 확인할 수 있습니다](https://langchain-ai.github.io/langgraphjs/how-tos/configuration/).



In [ ]:
actor_id = "user-1"
config = {
    "configurable": {
        "thread_id": "session-1",  # 필수: 내부적으로 Bedrock AgentCore session_id에 매핑됨
        "actor_id": actor_id,  # 필수: 내부적으로 Bedrock AgentCore actor_id에 매핑됨
    }
}

## 7단계: Agent 테스트

음식 선호도에 관한 대화를 통해 영양 도우미를 테스트해 보겠습니다. Agent는 나중에 회상하고 pattern을 분석할 수 있도록 대화를 episode로 자동 포착합니다.

In [ ]:
# 실행 중 agent 출력을 보기 좋게 표시하는 helper function
def run_agent(query: str, config: RunnableConfig):
    printed_ids = set()
    events = graph.stream(
        {"messages": [{"role": "user", "content": query}]},
        config,
        stream_mode="values",
    )
    for event in events:
        if "messages" in event:
            for msg in event["messages"]:
                # 이 message가 이미 출력되었는지 확인
                if id(msg) not in printed_ids:
                    msg.pretty_print()
                    printed_ids.add(id(msg))


prompt = """
Hey there! Im cooking one of my favorite meals tonight, salmon with rice and veggies (healthy). Has
great macros for my weightlifting competition that is coming up. What can I add to this dish to make it taste better
and also improve the protein and vitamins I get?
"""

run_agent(prompt, config)

### 무엇이 저장되었나요?
보시는 것처럼 model은 아직 이전 식단 계획 session에서 얻은 insight가 없습니다.

Pre/post model hook을 사용하는 이 구현에서는 두 개의 message가 저장되었습니다. 첫 번째 user message와 AI model의 응답이 모두 AgentCore Memory에 대화 event로 저장되었습니다. Episode와 reflection 생성에는 잠시 시간이 걸릴 수 있으므로 처음에 아무것도 검색되지 않으면 몇 분 후 다시 시도하세요.

이후 episodic strategy가 이 message들을 처리하여 AgentCore 장기 메모리에 구조화된 episode와 reflection을 생성합니다. 지금까지 무엇이 저장되었는지 store를 직접 확인해 보겠습니다.

In [ ]:
# 대화 message 검색
search_namespace = ("nutrition", actor_id, "session-1/")
result = store.search(search_namespace, query="meal", limit=3)
print(f"Conversation messages result: {result}")

In [ ]:
# LangGraph에서 episodic 장기 메모리를 검색하는 올바른 방법
from bedrock_agentcore.memory import MemoryClient

# Store가 아닌 memory client를 직접 사용
memory_client = MemoryClient(region_name=region)

print("=== Searching Long-Term Episodic Memories ===")
print(f"Memory ID: {memory_id}")
print()

# Episodic memory 검색 (episode)
print("1. Episodic namespace: /nutrition/user-1/session-1/")
try:
    episodes = memory_client.retrieve_memories(
        memory_id=memory_id,
        namespace="/nutrition/user-1/session-1/",
        query="meal",
        top_k=3,
    )
    print(f"   Found {len(episodes)} episode memories")
    for mem in episodes:
        content = mem.get("content", {})
        text = content.get("text", str(content))
        print(f"   - {text[:300]}...")
except Exception as e:
    print(f"   Error: {e}")
print()

# Reflection memory 검색
print("2. Reflection namespace: /nutrition/user-1/")
try:
    reflections = memory_client.retrieve_memories(
        memory_id=memory_id, namespace="/nutrition/user-1/", query="meal", top_k=3
    )
    print(f"   Found {len(reflections)} reflection memories")
    for mem in reflections:
        content = mem.get("content", {})
        text = content.get("text", str(content))
        print(f"   - {text[:300]}...")
except Exception as e:
    print(f"   Error: {e}")

### Agent의 store 액세스

**참고** - AgentCore Memory가 이러한 event를 background에서 처리하므로 메모리가 추출되고 장기 메모리 검색용 embedding이 생성되기까지 몇 분 정도 걸릴 수 있습니다.

좋습니다! 앞선 대화 message를 바탕으로 장기 메모리가 namespace에 추출된 것을 확인했습니다.

이제 새 session을 시작하고 저녁 메뉴 추천을 요청해 보겠습니다. Agent는 store를 통해 추출된 장기 메모리에 액세스하여 사용자가 좋아할 만한 메뉴를 추천할 수 있습니다.

In [ ]:
config = {
    "configurable": {
        "thread_id": "session-2",  # 새로운 session ID
        "actor_id": actor_id,  # 동일한 actor ID
    }
}

run_agent("Today's a new day, what should I make for dinner tonight?", config)

### 마무리

보시는 것처럼 agent의 대화는 자동으로 포착되어 turn 단위 분석이 포함된 구조화된 episode로 처리됩니다. Episodic strategy는 여러 식단 계획 session에서 insight를 생성하여 pattern을 식별하고 시간에 따른 선호도 변화를 추적합니다.

AgentCoreMemoryStore는 매우 유연하여 pre/post model hook을 사용하거나 store operation을 수행하는 tool만 사용하는 등 다양한 방식으로 구현할 수 있습니다. Checkpointing에 AgentCoreMemorySaver를 함께 사용하면 전체 대화 state와 episodic reflection을 결합하여 복잡하고 지능적인 agent system을 구성할 수 있습니다.